# Preliminary Analysis on Base Questions 1.1

This 1.1 Version of analysis is based on updated cleaned ridership datasets, 'cleaned_2019-01-03_data' and 'cleaned_2022-01_data'.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


## File loading

In [4]:
path_1 = 'C:\\Users\\Huihao Xing\\Documents\\临时文件\\G1\\DS 701\\Project\\Dataset\\cleaned_2019-01-03_data.csv'
path_2 = 'C:\\Users\\Huihao Xing\\Documents\\临时文件\\G1\\DS 701\\Project\\Dataset\\cleaned_2022-01_data'

In [5]:
df = pd.read_csv(path_1)

df

,service_date,route_id,direction,half_trip_id,stop_id,time_point_id,time_point_order,point_type,standard_type,scheduled,actual,scheduled_headway,headway,scheduled_datetime,actual_datetime,available_bus_depart_time
0,1/1/2019,111,Inbound,41928237,5605,belsq,5,Midpoint,Schedule,05:48:00,05:59:37,NaN,NaN,1/1/2019 05:48,1/1/2019 05:59,1/1/2019 05:59
1,1/1/2019,111,Inbound,41928270,5605,belsq,5,Midpoint,Headway,06:03:00,06:08:16,900.0,519.0,1/1/2019 06:03,1/1/2019 06:08,1/1/2019 06:08
2,1/1/2019,111,Inbound,41928283,5605,belsq,5,Midpoint,Headway,06:18:00,06:29:21,900.0,25.0,1/1/2019 06:18,1/1/2019 06:29,1/1/2019 06:27
3,1/1/2019,111,Inbound,41928251,5605,belsq,2,Midpoint,Headway,06:27:00,06:27:03,540.0,1127.0,1/1/2019 06:27,1/1/2019 06:27,1/1/2019 06:27
4,1/1/2019,111,Inbound,41927983,5605,belsq,5,Midpoint,Headway,06:33:00,06:28:56,360.0,113.0,1/1/2019 06:33,1/1/2019 06:28,1/1/2019 06:43
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1048570,3/16/2019,24,Outbound,42792244,574,rvcen,4,Midpoint,Schedule,01:08:00,01:56:11,NaN,NaN,3/17/2019 01:08,3/17/2019 01:56,3/17/2019 01:56
1048571,3/16/2019,24,Outbound,42792203,6368,rivcm,2,Midpoint,Schedule,05:51:00,05:50:26,NaN,NaN,3/16/2019 05:51,3/16/2019 05:50,3/16/2019 06:20
1048572,3/16/2019,24,Outbound,42792758,6368,rivcm,6,Midpoint,Schedule,06:15:00,06:20:45,NaN,NaN,3/16/2019 06:15,3/16/2019 06:20,3/16/2019 06:20
1048573,3/16/2019,24,Outbound,42792205,6368,rivcm,6,Midpoint,Schedule,06:55:00,06:54:29,NaN,NaN,3/16/2019 06:55,3/16/2019 06:54,3/16/2019 07:40


## Question 3

In [6]:
# mean waiting time
df['scheduled'] = pd.to_datetime(df['scheduled'], format='%H:%M:%S', errors='coerce')
df['actual'] = pd.to_datetime(df['actual'], format='%H:%M:%S', errors='coerce')

df['wait_time'] = (df['actual'] - df['scheduled']).dt.total_seconds()

df_filtered = df.dropna(subset=['wait_time'])

on_time = df_filtered[df_filtered['wait_time'] <= 0]['wait_time'].mean()  
delayed = df_filtered[df_filtered['wait_time'] > 0]['wait_time'].mean()  

print(f"Average waiting time for on-time buses: {abs(on_time)} seconds")
print(f"Average waiting time for delayed buses: {delayed} seconds")


Average waiting time for on-time buses: 643.8242727552334 seconds
Average waiting time for delayed buses: 335.79826366690526 seconds


In [7]:
df['scheduled'] = pd.to_datetime(df['scheduled'], format='%H:%M:%S', errors='coerce')
df['actual'] = pd.to_datetime(df['actual'], format='%H:%M:%S', errors='coerce')

# Calculating wait_time as the difference in seconds
df['wait_time'] = (df['actual'] - df['scheduled']).dt.total_seconds()

# Filtering to remove any NaN wait_time values
df_filtered = df.dropna(subset=['wait_time'])

# Redefining "on-time" as buses arriving within a 1-minute (60 seconds) window around scheduled time
on_time_refined = df_filtered[(df_filtered['wait_time'] >= -60) & (df_filtered['wait_time'] <= 60)]
delayed = df_filtered[df_filtered['wait_time'] > 60]  # Delayed as strictly over 1 minute

# Calculating mean waiting times
on_time_mean_wait = on_time_refined['wait_time'].mean()
delayed_mean_wait = delayed['wait_time'].mean()

on_time_mean_wait, delayed_mean_wait

(6.005597276646684, 392.9192985161267)

## Question 4

In [8]:
# mean delay - all routes

average_delay_all_routes = df_filtered[df_filtered['wait_time'] > 0]['wait_time'].mean()

average_delay_all_routes

335.79826366690526

## Question 5

In [10]:
# mean delay - taget routes
target_routes = ['22', '29', '15', '45', '28', '44', '42', '17', '23', '31', '26', '111', '24', '33', '14']

df_filtered['route_id'] = df_filtered['route_id'].astype(str)

target_route_data = df_filtered[df_filtered['route_id'].isin(target_routes)] 

average_delay_target_routes = target_route_data[target_route_data['wait_time'] > 0]['wait_time'].mean()

average_delay_target_routes

335.79826366690526

In [12]:
## Detecting Reason

df['route_id'].value_counts()

111    244777
23     181247
28     178188
22     129631
31     102932
15      96619
44      66701
45      61908
29      42501
42      34475
17      33014
26      27291
24      26893
14      25490
33      15166
Name: route_id, dtype: int64

routes 19 is missing from datasets

## Question 6

In [11]:
# disparity in service level
route_delays = df_filtered.groupby('route_id').agg(
    total_buses=('wait_time', 'count'),                       # Total number of buses per route
    delayed_buses=('wait_time', lambda x: (x > 0).sum())      # Number of delayed buses per route
).reset_index()

route_delays['delay_percentage'] = (route_delays['delayed_buses'] / route_delays['total_buses']) * 100

route_delays_sorted = route_delays.sort_values(by='delay_percentage', ascending=False)

print(route_delays_sorted.head(15)) 
print('\n')
print(route_delays_sorted.tail(15)) 

    route_id  total_buses  delayed_buses  delay_percentage
6         26        27291          24331         89.153934
4         23       181247         158634         87.523656
10        33        15166          13227         87.214823
8         29        42501          37034         87.136773
5         24        26893          22913         85.200610
9         31       102932          86579         84.112812
3         22       129631         108264         83.517060
12        44        66701          55287         82.887813
13        45        61908          48032         77.586095
2         17        33014          25503         77.249046
1         15        96619          73922         76.508761
11        42        34475          26041         75.535896
7         28       178188         124740         70.004714
0         14        25490          16302         63.954492
14       111       244777         130824         53.446198


    route_id  total_buses  delayed_buses  delay_percen